# 02 · Instruction tuning (supervised fine-tuning) on medical Q&A
Starts from the adapter trained in notebook 01 and teaches the model to answer as a GP-style assistant. Loss is computed on the answer only.

In [ ]:
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install -q -U "transformers>=5"      # Qwen3.5 needs transformers v5; restart the runtime if Colab asks, then skip this cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, zipfile
BASE = '/content/drive/MyDrive/health-llm'            # put colab_health_bundle.zip here
os.makedirs(BASE, exist_ok=True)
if not os.path.exists('cpt_train.jsonl'):
    zipfile.ZipFile(f'{BASE}/colab_health_bundle.zip').extractall('.')
sys.path.insert(0, '.')
MODEL = 'unsloth/Qwen3.5-4B-Base'    # base (pre-trained only) model, the right start for continued pre-training. T4 too slow / out of memory: 'unsloth/Qwen3.5-2B-Base'
LOAD = dict(load_in_4bit=False, load_in_16bit=True, full_finetuning=False)   # Unsloth advises against 4-bit on Qwen3.5
LORA = dict(r=32, lora_alpha=64, lora_dropout=0, use_gradient_checkpointing='unsloth', random_state=3407,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
!nvidia-smi -L

In [ ]:
import glob, json, torch, hl_lib
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import Trainer, TrainingArguments
from datasets import Dataset

MAX_SFT, MAX_LEN = 6000, 1536
model, tok = FastLanguageModel.from_pretrained(f'{BASE}/adapter_cpt', max_seq_length=2048, **LOAD)   # base + stage-1 LoRA
FastLanguageModel.for_training(model)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('trainable parameters:', n_train)
assert n_train > 0, 'LoRA weights are frozen after loading; see Unsloth docs on continuing training from a saved adapter'
sft = hl_lib.load_jsonl('sft_train.jsonl')[:MAX_SFT]; val = hl_lib.load_jsonl('sft_val.jsonl')[:100]
mk = lambda rows: Dataset.from_list([hl_lib.sft_tokenize(tok, r['messages'], MAX_LEN) for r in rows])
train_ds, val_ds = mk(sft), mk(val)
print(len(train_ds), 'train /', len(val_ds), 'val examples')
print(hl_lib.chatml(sft[0]['messages'])[:800])

In [ ]:
pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
args = TrainingArguments(output_dir=f'{BASE}/sft_ckpt', per_device_train_batch_size=1, gradient_accumulation_steps=16, per_device_eval_batch_size=1,
    learning_rate=1e-4, lr_scheduler_type='cosine', warmup_steps=10, num_train_epochs=1, optim='adamw_8bit', weight_decay=0.01,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=5, eval_strategy='steps', eval_steps=25,
    save_steps=25, save_total_limit=2, report_to='none', seed=3407, remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=lambda b: hl_lib.pad_collate(b, pad))
trainer.train(resume_from_checkpoint=bool(glob.glob(f'{BASE}/sft_ckpt/checkpoint-*')))
model.save_pretrained(f'{BASE}/adapter_sft'); tok.save_pretrained(f'{BASE}/adapter_sft')
json.dump(trainer.state.log_history, open(f'{BASE}/sft_log.json', 'w'))
print('saved', f'{BASE}/adapter_sft')

In [ ]:
import matplotlib.pyplot as plt
log = json.load(open(f'{BASE}/sft_log.json'))
tr = [(l['step'], l['loss']) for l in log if 'loss' in l]; ev = [(l['step'], l['eval_loss']) for l in log if 'eval_loss' in l]
plt.plot(*zip(*tr), label='train loss'); plt.plot(*zip(*ev), 'o-', label='validation loss'); plt.xlabel('optimizer step'); plt.ylabel('loss (answer tokens)')
plt.title('Instruction tuning'); plt.legend(); plt.grid(alpha=.3); plt.savefig(f'{BASE}/sft_loss.png', dpi=150); plt.show()